# Tower 17: Performance QA -- Solace Browser

**NORTHSTAR:** AI Worker Platform (Token Economics + Local-First + Evidence-Driven)  
**Tower:** 17 of 49 | Tower of Performance  
**DNA:** `performance(excellent) = speed(sub_second) x memory(bounded) x battery(efficient) x network(minimal) x render(smooth_60fps) x startup(instant) x scale(graceful)`  
**Target:** solace-browser (`/home/phuc/projects/solace-browser/`)  
**Floors covered:** 1-12 (Core Web Vitals, JS Perf, CSS Perf, Network, Startup, Memory, Battery, Rendering, Fonts, Images, API Response, Service Worker)  
**RUNG_TARGET:** 65537  
**Auth:** 65537

---

This notebook probes the solace-browser codebase for performance characteristics across the first 12 floors of Tower 17. Each cell maps to a specific floor and persona, checking real files on disk for performance patterns, anti-patterns, and measurable metrics.

In [ ]:
# Cell 1: Bootstrap -- imports, paths, helper functions
# Tower 17: Performance QA -- probing solace-browser for performance characteristics
import os, sys, re, json, hashlib, glob as globmod
from pathlib import Path
from datetime import datetime

NORTHSTAR = "performance-qa"
NOTEBOOK_PATH = "notebooks/qa/07-performance-qa.ipynb"
TOWER = 17
TOWER_NAME = "Tower of Performance"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER_ROOT / "web"
SRC = BROWSER_ROOT / "src"
DATA = BROWSER_ROOT / "data" / "default"
CSS_FILE = WEB / "css" / "site.css"
JS_DIR = WEB / "js"
VENDOR_DIR = WEB / "vendor"
IMAGES_DIR = WEB / "images"
THEMES_DIR = WEB / "css" / "themes"
LOCALES_DIR = BROWSER_ROOT / "app" / "locales"

results = []

def record(probe_id, name, passed, detail=""):
    """Record a PASS/FAIL result with evidence detail."""
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    """Safely read a file, returning empty string if missing."""
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

def file_exists(path):
    """Check if file exists at path."""
    return Path(path).exists()

def file_size_bytes(path):
    """Return file size in bytes, 0 if missing."""
    p = Path(path)
    return p.stat().st_size if p.exists() else 0

def file_size_kb(path):
    """Return file size in KB."""
    return round(file_size_bytes(path) / 1024, 1)

def collect_files(directory, pattern="*", recursive=True):
    """Collect all files matching pattern under directory."""
    d = Path(directory)
    if not d.exists():
        return []
    return list(d.rglob(pattern) if recursive else d.glob(pattern))

def search_in_files(directory, pattern, file_glob="*.html"):
    """Search for regex pattern in files matching glob. Return list of (filepath, matches)."""
    hits = []
    d = Path(directory)
    if not d.exists():
        return hits
    for f in d.rglob(file_glob):
        content = read_file(f)
        matches = re.findall(pattern, content, re.IGNORECASE)
        if matches:
            hits.append((str(f.relative_to(BROWSER_ROOT)), len(matches)))
    return hits

assert BROWSER_ROOT.exists(), f"solace-browser not found at {BROWSER_ROOT}"
print(f"Tower {TOWER}: {TOWER_NAME}")
print(f"Browser root: {BROWSER_ROOT}")
print(f"CSS file:     {CSS_FILE} ({file_size_kb(CSS_FILE)} KB)")
print(f"JS dir:       {JS_DIR} (exists={JS_DIR.exists()})")
print(f"Vendor dir:   {VENDOR_DIR} (exists={VENDOR_DIR.exists()})")
print(f"Images dir:   {IMAGES_DIR} (exists={IMAGES_DIR.exists()})")
print(f"Themes dir:   {THEMES_DIR} (exists={THEMES_DIR.exists()})")
print("Bootstrap complete.")

In [ ]:
# ============================================================
# Floor 1: Core Web Vitals Auditor
# Persona: audits LCP, FID/INP, CLS indicators in source
# Questions probed:
#   Q1: CSS file sizes -- site.css should be <100KB uncompressed
#   Q2: Are critical CSS and fonts preloaded to minimize LCP?
#   Q3: Are there render-blocking CSS patterns?
#   Q4: Does FOUC prevention exist (layout.js)?
#   Q5: Are there explicit width/height on images to prevent CLS?
# ============================================================

# Q1: CSS file sizes -- site.css should be <100KB uncompressed
css_size = file_size_bytes(CSS_FILE)
css_kb = round(css_size / 1024, 1)
theme_files = collect_files(THEMES_DIR, "*.css", recursive=False)
theme_sizes = {f.name: round(f.stat().st_size / 1024, 1) for f in theme_files}
total_css_kb = css_kb + sum(theme_sizes.values())
record("F1-Q1", "site.css < 100KB uncompressed", css_kb < 100,
       f"site.css={css_kb}KB, themes={theme_sizes}, total_css={total_css_kb}KB")

# Q2: Preload hints for critical CSS/fonts in HTML files
html_files = collect_files(WEB, "*.html", recursive=False)
preload_count = 0
for hf in html_files:
    content = read_file(hf)
    preloads = re.findall(r'<link[^>]*rel=["\']preload["\']', content, re.IGNORECASE)
    preload_count += len(preloads)
record("F1-Q2", "Preload hints for critical resources exist", preload_count > 0,
       f"Found {preload_count} <link rel=preload> across {len(html_files)} HTML files")

# Q3: Render-blocking CSS patterns (CSS without media or preload)
blocking_css = []
for hf in html_files:
    content = read_file(hf)
    # Find stylesheet links that are NOT preload and have no media=print pattern
    css_links = re.findall(r'<link[^>]*rel=["\']stylesheet["\'][^>]*>', content, re.IGNORECASE)
    for link in css_links:
        if 'media=' not in link.lower() and 'onload=' not in link.lower():
            # This is a potentially render-blocking stylesheet
            href = re.search(r'href=["\']([^"\']*)["\'\s]', link)
            href_val = href.group(1) if href else 'unknown'
            if 'site.css' not in href_val:  # site.css is expected to be blocking
                blocking_css.append((hf.name, href_val))
record("F1-Q3", "No unexpected render-blocking CSS", len(blocking_css) == 0,
       f"Found {len(blocking_css)} non-critical blocking stylesheets: {blocking_css[:5]}")

# Q4: FOUC prevention exists
layout_js = WEB / "js" / "layout.js"
layout_content = read_file(layout_js)
has_fouc_prevention = bool(re.search(
    r'fouc|flash|DOMContentLoaded|visibility|opacity.*0|hidden.*show',
    layout_content, re.IGNORECASE
))
record("F1-Q4", "FOUC prevention in layout.js", has_fouc_prevention,
       f"layout.js exists={layout_js.exists()}, FOUC pattern found={has_fouc_prevention}")

# Q5: Images have explicit width/height to prevent CLS
images_with_dimensions = 0
images_without_dimensions = 0
for hf in html_files:
    content = read_file(hf)
    img_tags = re.findall(r'<img[^>]*>', content, re.IGNORECASE)
    for img in img_tags:
        has_w = 'width=' in img.lower() or 'width:' in img.lower()
        has_h = 'height=' in img.lower() or 'height:' in img.lower()
        if has_w and has_h:
            images_with_dimensions += 1
        else:
            images_without_dimensions += 1
total_imgs = images_with_dimensions + images_without_dimensions
pct = round(images_with_dimensions / total_imgs * 100) if total_imgs > 0 else 0
record("F1-Q5", "Images have explicit dimensions (>=80%)", pct >= 80,
       f"{images_with_dimensions}/{total_imgs} images have width+height ({pct}%)")

In [ ]:
# ============================================================
# Floor 2: JavaScript Performance Engineer
# Persona: script execution and bundle optimization
# Questions probed:
#   Q1: JS bundle count and sizes (each <50KB for custom JS)
#   Q2: No synchronous script loading (defer/async on all scripts)
#   Q3: No document.write() calls
#   Q4: Vendor JS total size check
#   Q5: Are event listeners cleaned up (removeEventListener present)?
# ============================================================

# Q1: JS bundle count and sizes (custom JS each <50KB)
custom_js = collect_files(JS_DIR, "*.js", recursive=False)
js_sizes = {}
oversized_js = []
for f in custom_js:
    size_kb = round(f.stat().st_size / 1024, 1)
    js_sizes[f.name] = size_kb
    if size_kb > 50:
        oversized_js.append((f.name, size_kb))
record("F2-Q1", "Custom JS files each <50KB", len(oversized_js) == 0,
       f"{len(custom_js)} files, oversized (>50KB): {oversized_js if oversized_js else 'none'}, sizes: {js_sizes}")

# Q2: No synchronous script loading (defer/async on all scripts)
sync_scripts = []
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    script_tags = re.findall(r'<script[^>]*src=[^>]*>', content, re.IGNORECASE)
    for tag in script_tags:
        has_defer = 'defer' in tag.lower()
        has_async = 'async' in tag.lower()
        if not has_defer and not has_async:
            src = re.search(r'src=["\']([^"\']*)["\'\s]', tag)
            src_val = src.group(1) if src else 'unknown'
            sync_scripts.append((str(hf.relative_to(BROWSER_ROOT)), src_val))
record("F2-Q2", "All scripts use defer or async", len(sync_scripts) == 0,
       f"Found {len(sync_scripts)} sync scripts: {sync_scripts[:10]}")

# Q3: No document.write() calls in any JS
doc_write_hits = []
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    if 'document.write' in content:
        count = content.count('document.write')
        doc_write_hits.append((str(f.relative_to(BROWSER_ROOT)), count))
record("F2-Q3", "No document.write() calls", len(doc_write_hits) == 0,
       f"Found in: {doc_write_hits if doc_write_hits else 'none'}")

# Q4: Vendor JS total size audit
vendor_js = collect_files(VENDOR_DIR, "*.js", recursive=True)
vendor_sizes = {}
total_vendor_kb = 0
for f in vendor_js:
    size_kb = round(f.stat().st_size / 1024, 1)
    vendor_sizes[f.name] = size_kb
    total_vendor_kb += size_kb
record("F2-Q4", "Vendor JS total < 2MB", total_vendor_kb < 2048,
       f"{len(vendor_js)} vendor files, total={round(total_vendor_kb, 1)}KB, files: {vendor_sizes}")

# Q5: Event listeners cleaned up (removeEventListener present in code)
js_all_files = collect_files(WEB, "*.js", recursive=True)
add_listener_count = 0
remove_listener_count = 0
for f in js_all_files:
    content = read_file(f)
    add_listener_count += len(re.findall(r'addEventListener', content))
    remove_listener_count += len(re.findall(r'removeEventListener', content))
ratio = round(remove_listener_count / add_listener_count * 100) if add_listener_count > 0 else 0
record("F2-Q5", "Event listeners have cleanup (removeEventListener >= 10%)",
       ratio >= 10,
       f"addEventListener={add_listener_count}, removeEventListener={remove_listener_count}, ratio={ratio}%")

In [ ]:
# ============================================================
# Floor 3: CSS Performance Specialist
# Persona: stylesheet optimization
# Questions probed:
#   Q1: CSS specificity -- no !important overuse (<20 instances)
#   Q2: No deep CSS nesting (selectors >4 levels)
#   Q3: CSS custom properties used efficiently (not duplicated)
#   Q4: CSS animations use GPU-friendly properties (transform/opacity)
#   Q5: No universal selectors in complex rules (* with combinators)
# ============================================================

css_content = read_file(CSS_FILE)

# Q1: CSS specificity -- no !important overuse (<20 instances)
important_matches = re.findall(r'!important', css_content)
important_count = len(important_matches)
record("F3-Q1", "CSS !important usage <20 instances", important_count < 20,
       f"Found {important_count} !important declarations in site.css")

# Q2: Deep CSS nesting check (selectors with >4 space-separated parts)
# Parse CSS selectors by looking for rules
css_rules = re.findall(r'([^{}]+)\{', css_content)
deep_selectors = []
for rule in css_rules:
    # Split by comma for grouped selectors
    for selector in rule.split(','):
        selector = selector.strip()
        # Skip @-rules, empty, and CSS comments
        if selector.startswith('@') or not selector:
            continue
        if selector.startswith('/*') or selector.startswith('*') or '===' in selector:
            continue
        if re.match(r'^[\s/*=\-]+$', selector):
            continue
        # Count depth by spaces (combinators)
        parts = selector.split()
        if len(parts) > 4:
            deep_selectors.append(selector[:80])
record("F3-Q2", "No excessively deep CSS selectors (>4 levels)", len(deep_selectors) < 10,
       f"Found {len(deep_selectors)} selectors >4 levels deep (first 5): {deep_selectors[:5]}")

# Q3: CSS custom properties check
custom_props_defined = re.findall(r'--[\w-]+\s*:', css_content)
custom_props_used = re.findall(r'var\(--[\w-]+', css_content)
record("F3-Q3", "CSS custom properties are used (>10 definitions)",
       len(custom_props_defined) > 10,
       f"Defined: {len(custom_props_defined)}, Used: {len(custom_props_used)}")

# Q4: CSS animations use GPU-friendly properties
all_css = css_content
for theme_f in collect_files(THEMES_DIR, "*.css", recursive=False):
    all_css += read_file(theme_f)

# Find animation/transition properties
animation_props = re.findall(r'(?:animation|transition)[^;]*;', all_css, re.IGNORECASE)
gpu_friendly = 0
non_gpu = []
for prop in animation_props:
    if re.search(r'transform|opacity|translate|scale|rotate', prop, re.IGNORECASE):
        gpu_friendly += 1
    elif re.search(r'width|height|top|left|right|bottom|margin|padding', prop, re.IGNORECASE):
        non_gpu.append(prop.strip()[:60])
total_anim = len(animation_props)
gpu_pct = round(gpu_friendly / total_anim * 100) if total_anim > 0 else 100
record("F3-Q4", "CSS animations are GPU-friendly (>=70%)", gpu_pct >= 70,
       f"{gpu_friendly}/{total_anim} GPU-friendly ({gpu_pct}%), non-GPU: {non_gpu[:5]}")

# Q5: No universal selectors in complex rules
universal_complex = re.findall(r'[\w\]\)][\s>+~]+\*[^/]', css_content)
record("F3-Q5", "No universal selectors in complex rules", len(universal_complex) < 5,
       f"Found {len(universal_complex)} complex universal selector patterns (first 3): {[u.strip()[:40] for u in universal_complex[:3]]}")

In [ ]:
# ============================================================
# Floor 4: Network Optimizer
# Persona: HTTP requests and data transfer
# Questions probed:
#   Q1: HTTP/2 push hints or preload links present
#   Q2: Cache-related headers or meta tags
#   Q3: No unnecessary polyfill loads
#   Q4: Assets use cache-busting (hash or version in filenames)
#   Q5: Link prefetch/preconnect hints present
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)

# Q1: Preload / HTTP2 push hints
preload_links = []
for hf in html_files:
    content = read_file(hf)
    preloads = re.findall(r'<link[^>]*rel=["\'](?:preload|prefetch|preconnect|dns-prefetch)["\'][^>]*>', content, re.IGNORECASE)
    for p in preloads:
        preload_links.append((str(hf.relative_to(BROWSER_ROOT)), p[:80]))
record("F4-Q1", "Preload/prefetch/preconnect hints present", len(preload_links) > 0,
       f"Found {len(preload_links)} resource hints: {preload_links[:5]}")

# Q2: Cache-related meta tags or server hints
cache_hints = []
for hf in html_files:
    content = read_file(hf)
    cache_meta = re.findall(r'<meta[^>]*(?:cache|pragma|expires)[^>]*>', content, re.IGNORECASE)
    if cache_meta:
        cache_hints.extend(cache_meta)
# Also check http_server files for cache headers
server_files = collect_files(SRC, "*.py", recursive=True) + collect_files(SRC, "*.js", recursive=True)
cache_header_files = []
for sf in server_files:
    content = read_file(sf)
    if re.search(r'Cache-Control|ETag|Last-Modified|max-age', content, re.IGNORECASE):
        cache_header_files.append(str(sf.relative_to(BROWSER_ROOT)))
record("F4-Q2", "Cache strategy exists (meta tags or server headers)",
       len(cache_hints) > 0 or len(cache_header_files) > 0,
       f"Meta cache tags: {len(cache_hints)}, Server files with cache headers: {cache_header_files[:5]}")

# Q3: No unnecessary polyfill loads
polyfill_loads = []
for hf in html_files:
    content = read_file(hf)
    polyfills = re.findall(r'(?:polyfill|core-js|babel-polyfill|es[56]-shim)', content, re.IGNORECASE)
    if polyfills:
        polyfill_loads.append((str(hf.relative_to(BROWSER_ROOT)), polyfills))
record("F4-Q3", "No unnecessary polyfill loads", len(polyfill_loads) == 0,
       f"Found polyfill references in: {polyfill_loads if polyfill_loads else 'none'}")

# Q4: Cache-busting in asset references (hash or version query params)
cache_busted = 0
not_cache_busted = 0
for hf in html_files:
    content = read_file(hf)
    asset_refs = re.findall(r'(?:src|href)=["\']([^"\']*)["\'\s]', content)
    for ref in asset_refs:
        if re.search(r'\.(js|css)$', ref):
            if '?' in ref or re.search(r'[a-f0-9]{8,}', ref):
                cache_busted += 1
            else:
                not_cache_busted += 1
total_assets = cache_busted + not_cache_busted
record("F4-Q4", "Static assets use cache-busting", cache_busted > not_cache_busted,
       f"{cache_busted}/{total_assets} assets have cache-busting (hash/version)")

# Q5: DNS prefetch or preconnect for external domains
external_preconnect = []
for hf in html_files:
    content = read_file(hf)
    hints = re.findall(r'<link[^>]*rel=["\'](?:preconnect|dns-prefetch)["\'][^>]*>', content, re.IGNORECASE)
    external_preconnect.extend(hints)
record("F4-Q5", "DNS prefetch/preconnect for external domains",
       len(external_preconnect) > 0,
       f"Found {len(external_preconnect)} preconnect/dns-prefetch hints")

In [ ]:
# ============================================================
# Floor 5: Startup Time Engineer
# Persona: application launch and cold start
# Questions probed:
#   Q1: Theme.js is small and loads first (critical path)
#   Q2: Layout.js deferred and not blocking
#   Q3: No heavy initialization in inline <script> blocks
#   Q4: Splash screen or loading state exists
#   Q5: First-run (OOBE) is lightweight
# ============================================================

# Q1: theme.js is small (<10KB) -- it's on the critical path for FOUC
theme_js = WEB / "js" / "theme.js"
theme_js_kb = file_size_kb(theme_js)
record("F5-Q1", "theme.js is small (<10KB, critical path)", theme_js_kb < 10,
       f"theme.js = {theme_js_kb}KB")

# Q2: layout.js uses defer
def check_script_defer(filename):
    """Check if a script is loaded with defer across all HTML files."""
    deferred = 0
    not_deferred = 0
    for hf in collect_files(WEB, "*.html", recursive=True):
        content = read_file(hf)
        pattern = rf'<script[^>]*src=["\'][^"\']*/{ re.escape(filename) }["\'][^>]*>'
        tags = re.findall(pattern, content, re.IGNORECASE)
        for tag in tags:
            if 'defer' in tag.lower() or 'async' in tag.lower():
                deferred += 1
            else:
                not_deferred += 1
    return deferred, not_deferred

deferred, not_deferred = check_script_defer("layout.js")
record("F5-Q2", "layout.js always loaded with defer",
       not_deferred == 0 and deferred > 0,
       f"deferred={deferred}, not_deferred={not_deferred}")

# Q3: No heavy inline scripts (inline <script> blocks >500 chars without comments)
heavy_inline = []
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    inline_scripts = re.findall(r'<script(?![^>]*src=)[^>]*>(.*?)</script>', content, re.DOTALL | re.IGNORECASE)
    for script in inline_scripts:
        # Strip comments and whitespace
        code = re.sub(r'/\*.*?\*/', '', script, flags=re.DOTALL)
        code = re.sub(r'//[^\n]*', '', code)
        code = code.strip()
        if len(code) > 500:
            heavy_inline.append((str(hf.relative_to(BROWSER_ROOT)), len(code)))
record("F5-Q3", "No heavy inline scripts (>500 chars of code)",
       len(heavy_inline) == 0,
       f"Found {len(heavy_inline)} heavy inline scripts: {heavy_inline[:5]}")

# Q4: Splash screen / loading state exists
splash_files = collect_files(IMAGES_DIR / "splash", "*", recursive=False)
has_splash_css = bool(re.search(r'splash|loading-screen|app-loading', read_file(CSS_FILE), re.IGNORECASE))
record("F5-Q4", "Splash screen or loading state exists",
       len(splash_files) > 0 or has_splash_css,
       f"Splash images: {len(splash_files)}, CSS splash class: {has_splash_css}")

# Q5: First-run (OOBE / onboarding) is lightweight
onboarding_js = WEB / "js" / "onboarding.js"
onboarding_kb = file_size_kb(onboarding_js)
start_html = WEB / "start.html"
start_exists = start_html.exists()
record("F5-Q5", "Onboarding is lightweight (<30KB JS)",
       onboarding_kb < 30,
       f"onboarding.js={onboarding_kb}KB, start.html exists={start_exists}")

In [ ]:
# ============================================================
# Floor 6: Memory Profiler
# Persona: RAM usage and memory management
# Questions probed:
#   Q1: DOM depth check -- no elements nested >15 levels (static analysis)
#   Q2: No setInterval without clearInterval (potential memory leak)
#   Q3: Event listener cleanup patterns present
#   Q4: No global variable pollution
#   Q5: Large data structures are bounded (not infinite growth)
# ============================================================

# Q1: DOM depth check -- estimate max nesting from HTML indentation
max_depth = 0
deepest_file = ""
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    depth = 0
    max_file_depth = 0
    for line in content.split('\n'):
        opens = len(re.findall(r'<(?!/)(?!br|hr|img|input|meta|link)[a-z][^>]*/?>(?!</)', line, re.IGNORECASE))
        closes = len(re.findall(r'</[a-z]', line, re.IGNORECASE))
        depth += opens - closes
        if depth > max_file_depth:
            max_file_depth = depth
    if max_file_depth > max_depth:
        max_depth = max_file_depth
        deepest_file = str(hf.relative_to(BROWSER_ROOT))
record("F6-Q1", "DOM depth <= 15 levels", max_depth <= 15,
       f"Max nesting depth: {max_depth} in {deepest_file}")

# Q2: setInterval without matching clearInterval
js_files = collect_files(WEB, "*.js", recursive=True)
set_interval_count = 0
clear_interval_count = 0
set_timeout_count = 0
clear_timeout_count = 0
for f in js_files:
    content = read_file(f)
    set_interval_count += len(re.findall(r'setInterval\s*\(', content))
    clear_interval_count += len(re.findall(r'clearInterval\s*\(', content))
    set_timeout_count += len(re.findall(r'setTimeout\s*\(', content))
    clear_timeout_count += len(re.findall(r'clearTimeout\s*\(', content))
# Allow some setIntervals without clear (e.g., for heartbeats)
uncleared_ratio = (set_interval_count - clear_interval_count) if set_interval_count > 0 else 0
record("F6-Q2", "setInterval has matching clearInterval",
       uncleared_ratio <= 3,
       f"setInterval={set_interval_count}, clearInterval={clear_interval_count}, "
       f"setTimeout={set_timeout_count}, clearTimeout={clear_timeout_count}")

# Q3: Event listener cleanup patterns
add_count = 0
remove_count = 0
for f in js_files:
    content = read_file(f)
    add_count += len(re.findall(r'\.addEventListener\(', content))
    remove_count += len(re.findall(r'\.removeEventListener\(', content))
record("F6-Q3", "removeEventListener calls present (>0)",
       remove_count > 0,
       f"addEventListener={add_count}, removeEventListener={remove_count}")

# Q4: No excessive global variable pollution
global_vars = set()
for f in collect_files(JS_DIR, "*.js", recursive=False):
    content = read_file(f)
    # Look for window.X = or var X at top level (heuristic)
    window_vars = re.findall(r'window\.(\w+)\s*=', content)
    global_vars.update(window_vars)
    # Also look for bare var/let/const at indent level 0
    top_level = re.findall(r'^(?:var|let|const)\s+(\w+)', content, re.MULTILINE)
    global_vars.update(top_level)
record("F6-Q4", "Global variable count reasonable (<30)",
       len(global_vars) < 30,
       f"Found {len(global_vars)} potential globals: {sorted(global_vars)[:15]}")

# Q5: JSON data files are bounded (evidence, oracle, config not unbounded)
data_json_files = collect_files(DATA, "*.json", recursive=True)
large_json = []
for f in data_json_files:
    size_kb = round(f.stat().st_size / 1024, 1)
    if size_kb > 500:
        large_json.append((str(f.relative_to(BROWSER_ROOT)), size_kb))
record("F6-Q5", "No unbounded JSON data files (>500KB)",
       len(large_json) == 0,
       f"Large JSON files: {large_json if large_json else 'none'}")

In [ ]:
# ============================================================
# Floor 7: Battery Life Engineer
# Persona: power consumption and efficiency
# Questions probed:
#   Q1: No busy-wait loops (while true / polling without sleep)
#   Q2: Animations respect prefers-reduced-motion
#   Q3: requestAnimationFrame used instead of setInterval for animations
#   Q4: No continuous polling (fetch in setInterval)
#   Q5: CSS animations are efficient (no paint storms)
# ============================================================

# Q1: No busy-wait loops
busy_wait = []
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    # Check for while(true) or while(1) without async/await
    busy = re.findall(r'while\s*\(\s*(?:true|1)\s*\)', content)
    if busy:
        busy_wait.append((str(f.relative_to(BROWSER_ROOT)), len(busy)))
record("F7-Q1", "No busy-wait loops in JS", len(busy_wait) == 0,
       f"Busy-wait patterns: {busy_wait if busy_wait else 'none'}")

# Q2: prefers-reduced-motion respected
all_css_content = read_file(CSS_FILE)
for tf in collect_files(THEMES_DIR, "*.css"):
    all_css_content += read_file(tf)
reduced_motion = re.findall(r'prefers-reduced-motion', all_css_content)
# Also check JS
js_reduced_motion = 0
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    js_reduced_motion += len(re.findall(r'prefers-reduced-motion|matchMedia.*reduce', content))
record("F7-Q2", "prefers-reduced-motion respected (CSS or JS)",
       len(reduced_motion) > 0 or js_reduced_motion > 0,
       f"CSS references: {len(reduced_motion)}, JS references: {js_reduced_motion}")

# Q3: requestAnimationFrame used instead of setInterval for animations
raf_count = 0
set_interval_anim = 0
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    raf_count += len(re.findall(r'requestAnimationFrame', content))
    # Check for setInterval with animation-like callbacks
    interval_calls = re.findall(r'setInterval\s*\([^)]*(?:animate|move|slide|rotate|fade|scroll)[^)]*\)', content, re.IGNORECASE)
    set_interval_anim += len(interval_calls)
record("F7-Q3", "requestAnimationFrame used for animations (not setInterval)",
       raf_count > 0 and set_interval_anim == 0,
       f"requestAnimationFrame={raf_count}, setInterval-for-animation={set_interval_anim}")

# Q4: No continuous polling (fetch/XMLHttpRequest in setInterval)
polling_patterns = []
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    # Look for fetch or XHR inside setInterval
    intervals = re.findall(r'setInterval\s*\([^)]*(?:fetch|XMLHttpRequest|\$\.ajax|\$\.get|\$\.post)[^)]*\)', content, re.IGNORECASE | re.DOTALL)
    if intervals:
        polling_patterns.append(str(f.relative_to(BROWSER_ROOT)))
record("F7-Q4", "No continuous network polling via setInterval",
       len(polling_patterns) == 0,
       f"Files with fetch-in-setInterval: {polling_patterns if polling_patterns else 'none'}")

# Q5: CSS animations use efficient properties (no layout triggers)
css_keyframes = re.findall(r'@keyframes[^{]*\{([^}]*(?:\{[^}]*\})*[^}]*)\}', all_css_content, re.DOTALL)
layout_in_keyframes = []
for kf in css_keyframes:
    layout_props = re.findall(r'(?:width|height|top|left|right|bottom|margin|padding)\s*:', kf)
    if layout_props:
        layout_in_keyframes.extend(layout_props)
record("F7-Q5", "CSS @keyframes avoid layout-triggering properties",
       len(layout_in_keyframes) == 0,
       f"Layout properties in @keyframes: {len(layout_in_keyframes)} -- {layout_in_keyframes[:5]}")

In [ ]:
# ============================================================
# Floor 8: Rendering Pipeline Expert
# Persona: paint and composite performance
# Questions probed:
#   Q1: will-change or transform3d hints for animated elements
#   Q2: No forced synchronous layout (read then write patterns)
#   Q3: CSS contain property used for performance isolation
#   Q4: No excessive box-shadow on frequently repainted elements
#   Q5: SVGs used for icons (not raster at wrong size)
# ============================================================

all_css = read_file(CSS_FILE)
for tf in collect_files(THEMES_DIR, "*.css"):
    all_css += read_file(tf)

# Q1: will-change or translateZ(0) hints for animated elements
will_change_count = len(re.findall(r'will-change', all_css, re.IGNORECASE))
translatez_count = len(re.findall(r'translateZ\s*\(\s*0', all_css, re.IGNORECASE))
translate3d_count = len(re.findall(r'translate3d', all_css, re.IGNORECASE))
total_gpu_hints = will_change_count + translatez_count + translate3d_count
record("F8-Q1", "GPU compositing hints present (will-change/translateZ)",
       total_gpu_hints > 0,
       f"will-change={will_change_count}, translateZ(0)={translatez_count}, translate3d={translate3d_count}")

# Q2: Forced synchronous layout patterns in JS
# These read layout (offsetHeight, getBoundingClientRect) then immediately write (style changes)
fsl_patterns = []
for f in collect_files(WEB, "*.js", recursive=True):
    content = read_file(f)
    # Look for offsetHeight/Width, getBoundingClientRect, clientHeight etc.
    reads = re.findall(r'(?:offset(?:Height|Width|Top|Left)|getBoundingClientRect|client(?:Height|Width)|scroll(?:Height|Width|Top|Left))', content)
    if len(reads) > 5:  # Heuristic: many layout reads suggest potential thrashing
        fsl_patterns.append((str(f.relative_to(BROWSER_ROOT)), len(reads)))
record("F8-Q2", "No excessive layout reads in single files (>5 per file)",
       len(fsl_patterns) < 3,
       f"Files with many layout reads: {fsl_patterns if fsl_patterns else 'none'}")

# Q3: CSS contain property for performance isolation
contain_count = len(re.findall(r'contain\s*:', all_css))
record("F8-Q3", "CSS contain property used for isolation",
       contain_count > 0,
       f"Found {contain_count} 'contain:' declarations")

# Q4: Box-shadow count (excessive shadows can hurt paint performance)
box_shadows = re.findall(r'box-shadow\s*:', all_css)
record("F8-Q4", "Box-shadow usage reasonable (<30 instances)",
       len(box_shadows) < 30,
       f"Found {len(box_shadows)} box-shadow declarations")

# Q5: SVGs used for icons
svg_icons = collect_files(WEB, "*.svg", recursive=True)
favicon_svg = (WEB / "favicon.svg").exists()
inline_svg = 0
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    inline_svg += len(re.findall(r'<svg[\s>]', content, re.IGNORECASE))
record("F8-Q5", "SVGs used for icons/illustrations",
       len(svg_icons) > 0 or inline_svg > 0,
       f"SVG files: {len(svg_icons)}, favicon.svg={favicon_svg}, inline SVGs in HTML: {inline_svg}")

In [ ]:
# ============================================================
# Floor 9: Font Loading Specialist
# Persona: typography performance
# Questions probed:
#   Q1: Web font loading strategy (font-display: swap)
#   Q2: Font file count and total size
#   Q3: System font stack fallback defined
#   Q4: Fonts preloaded in document head
#   Q5: No excessive font variants loaded
# ============================================================

all_css = read_file(CSS_FILE)
for tf in collect_files(THEMES_DIR, "*.css"):
    all_css += read_file(tf)

# Q1: font-display: swap used for web fonts
font_display = re.findall(r'font-display\s*:\s*(\w+)', all_css, re.IGNORECASE)
has_swap = any(d.lower() == 'swap' for d in font_display)
record("F9-Q1", "font-display: swap for web fonts",
       has_swap or len(font_display) == 0,  # Pass if no custom fonts or swap is used
       f"font-display values: {font_display if font_display else 'none (system fonts only)'}")

# Q2: Font file count and size
font_files = collect_files(WEB, "*.woff2", recursive=True) + \
             collect_files(WEB, "*.woff", recursive=True) + \
             collect_files(WEB, "*.ttf", recursive=True) + \
             collect_files(WEB, "*.otf", recursive=True) + \
             collect_files(WEB, "*.eot", recursive=True)
total_font_kb = sum(round(f.stat().st_size / 1024, 1) for f in font_files)
record("F9-Q2", "Web font total size < 500KB",
       total_font_kb < 500,
       f"{len(font_files)} font files, total={total_font_kb}KB")

# Q3: System font stack fallback
system_fonts = re.findall(
    r'font-family[^;]*(?:system-ui|-apple-system|BlinkMacSystemFont|Segoe UI|sans-serif)',
    all_css, re.IGNORECASE
)
record("F9-Q3", "System font stack fallback defined",
       len(system_fonts) > 0,
       f"Found {len(system_fonts)} system font stack declarations")

# Q4: Font preloading in HTML head
font_preloads = []
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    fp = re.findall(r'<link[^>]*rel=["\']preload["\'][^>]*as=["\']font["\']', content, re.IGNORECASE)
    font_preloads.extend(fp)
record("F9-Q4", "Font files preloaded in HTML head",
       len(font_preloads) > 0 or len(font_files) == 0,  # Pass if no fonts or preloaded
       f"Font preload links: {len(font_preloads)}, font files: {len(font_files)}")

# Q5: No excessive font variants (>6 weight/style combos)
font_face_rules = re.findall(r'@font-face\s*\{[^}]*\}', all_css, re.DOTALL | re.IGNORECASE)
record("F9-Q5", "Font variant count reasonable (<=6 @font-face rules)",
       len(font_face_rules) <= 6,
       f"Found {len(font_face_rules)} @font-face rules")

In [ ]:
# ============================================================
# Floor 10: Image Optimization Expert
# Persona: image format, size, and loading
# Questions probed:
#   Q1: No uncompressed PNGs > 100KB
#   Q2: Lazy loading for images (loading=lazy attribute)
#   Q3: Image dimensions specified (width/height attributes)
#   Q4: Modern image formats used (WebP, AVIF, SVG)
#   Q5: Total image payload reasonable (<5MB)
# ============================================================

# Q1: No uncompressed PNGs > 100KB
large_pngs = []
all_images = collect_files(IMAGES_DIR, "*", recursive=True)
for img in all_images:
    if img.suffix.lower() == '.png':
        size_kb = round(img.stat().st_size / 1024, 1)
        if size_kb > 100:
            large_pngs.append((str(img.relative_to(BROWSER_ROOT)), size_kb))
record("F10-Q1", "No PNGs > 100KB", len(large_pngs) == 0,
       f"Found {len(large_pngs)} oversized PNGs: {large_pngs[:5]}")

# Q2: Lazy loading for images
lazy_count = 0
non_lazy_count = 0
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    img_tags = re.findall(r'<img[^>]*>', content, re.IGNORECASE)
    for img in img_tags:
        if 'loading="lazy"' in img.lower() or "loading='lazy'" in img.lower():
            lazy_count += 1
        else:
            non_lazy_count += 1
total_img_tags = lazy_count + non_lazy_count
lazy_pct = round(lazy_count / total_img_tags * 100) if total_img_tags > 0 else 0
record("F10-Q2", "Images use lazy loading (>=50%)", lazy_pct >= 50,
       f"{lazy_count}/{total_img_tags} images have loading=lazy ({lazy_pct}%)")

# Q3: Image dimensions specified
with_dims = 0
without_dims = 0
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    img_tags = re.findall(r'<img[^>]*>', content, re.IGNORECASE)
    for img in img_tags:
        has_w = bool(re.search(r'\bwidth\s*=', img, re.IGNORECASE))
        has_h = bool(re.search(r'\bheight\s*=', img, re.IGNORECASE))
        if has_w and has_h:
            with_dims += 1
        else:
            without_dims += 1
total = with_dims + without_dims
dim_pct = round(with_dims / total * 100) if total > 0 else 100
record("F10-Q3", "Image dimensions specified (>=70%)", dim_pct >= 70,
       f"{with_dims}/{total} images have width+height ({dim_pct}%)")

# Q4: Modern image formats used (WebP, AVIF, SVG present)
webp_files = collect_files(IMAGES_DIR, "*.webp", recursive=True)
avif_files = collect_files(IMAGES_DIR, "*.avif", recursive=True)
svg_files = collect_files(WEB, "*.svg", recursive=True)
has_modern = len(webp_files) > 0 or len(avif_files) > 0 or len(svg_files) > 0
record("F10-Q4", "Modern image formats present (WebP/AVIF/SVG)",
       has_modern,
       f"WebP: {len(webp_files)}, AVIF: {len(avif_files)}, SVG: {len(svg_files)}")

# Q5: Total image payload
total_img_bytes = 0
img_by_type = {}
for img in all_images:
    ext = img.suffix.lower()
    size = img.stat().st_size
    total_img_bytes += size
    img_by_type[ext] = img_by_type.get(ext, 0) + size
total_img_mb = round(total_img_bytes / (1024 * 1024), 2)
type_summary = {k: f"{round(v/1024, 1)}KB" for k, v in sorted(img_by_type.items())}
record("F10-Q5", "Total image payload < 5MB", total_img_mb < 5,
       f"Total: {total_img_mb}MB, by type: {type_summary}")

In [ ]:
# ============================================================
# Floor 11: API Response Time Auditor
# Persona: backend API performance
# Questions probed:
#   Q1: Server code uses async patterns (not blocking)
#   Q2: API responses are minimal (no verbose formatting)
#   Q3: Server serves static files efficiently
#   Q4: No unnecessary middleware on static routes
#   Q5: HTTP server startup code is lightweight
# ============================================================

# Locate server files
server_py_files = collect_files(SRC, "*.py", recursive=True)
http_server_files = [f for f in server_py_files if 'server' in f.name.lower() or 'http' in f.name.lower()]

# Also check for Node.js server
server_js_files = collect_files(SRC, "*.js", recursive=True) + [f for f in collect_files(BROWSER_ROOT, "*.js", recursive=False)]
server_js_files = [f for f in server_js_files if 'server' in f.name.lower() or 'http' in f.name.lower()]

all_server_files = http_server_files + server_js_files
all_server_content = ""
for f in all_server_files:
    all_server_content += read_file(f) + "\n"

# Q1: Async patterns used
async_patterns = len(re.findall(r'(?:async\s+def|async\s+function|await\s|asyncio|aiohttp)', all_server_content, re.IGNORECASE))
record("F11-Q1", "Server uses async patterns",
       async_patterns > 0 or len(all_server_files) == 0,
       f"Async patterns: {async_patterns} in {len(all_server_files)} server files: {[f.name for f in all_server_files]}")

# Q2: JSON responses are compact (no pretty-print in production)
pretty_json = re.findall(r'json\.dumps\([^)]*indent', all_server_content)
jsonify_indent = re.findall(r'jsonify|JSON\.stringify\([^)]*null.*[2-4]', all_server_content)
record("F11-Q2", "API responses are compact (no pretty-print indent)",
       len(pretty_json) == 0,
       f"json.dumps with indent: {len(pretty_json)}, jsonify: {len(jsonify_indent)}")

# Q3: Static file serving check
static_serving = bool(re.search(r'static|serve_file|send_file|sendFile|StaticFiles', all_server_content, re.IGNORECASE))
record("F11-Q3", "Static file serving configured",
       static_serving or len(all_server_files) == 0,
       f"Static file serving found: {static_serving}")

# Q4: MIME type handling for static assets
mime_handling = bool(re.search(r'content.type|mimetype|mime_type|MIME|guess_type', all_server_content, re.IGNORECASE))
record("F11-Q4", "MIME type handling present",
       mime_handling or len(all_server_files) == 0,
       f"MIME handling found: {mime_handling}")

# Q5: Server code is lightweight (< 500 lines per file)
heavy_server = []
for f in all_server_files:
    lines = len(read_file(f).split('\n'))
    if lines > 500:
        heavy_server.append((f.name, lines))
record("F11-Q5", "Server files are lean (<500 lines each)",
       len(heavy_server) == 0,
       f"Heavy server files: {heavy_server if heavy_server else 'none (or no server files)'}")

In [ ]:
# ============================================================
# Floor 12: Service Worker / Cache Expert
# Persona: offline caching and PWA performance
# Questions probed:
#   Q1: Service worker file exists (sw.js / service-worker.js)
#   Q2: Cache-first strategy for static assets
#   Q3: Manifest.json for PWA installability
#   Q4: Offline fallback page exists
#   Q5: Cache versioning / invalidation strategy
# ============================================================

# Q1: Service worker file exists
sw_paths = [
    WEB / "sw.js",
    WEB / "service-worker.js",
    WEB / "serviceworker.js",
    BROWSER_ROOT / "sw.js",
    BROWSER_ROOT / "service-worker.js",
]
sw_found = None
for p in sw_paths:
    if p.exists():
        sw_found = p
        break
# Also check if any HTML registers a service worker
sw_register = []
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    if 'serviceWorker' in content or 'service-worker' in content:
        sw_register.append(str(hf.relative_to(BROWSER_ROOT)))
record("F12-Q1", "Service worker exists",
       sw_found is not None or len(sw_register) > 0,
       f"SW file: {sw_found}, HTML registrations: {sw_register}")

# Q2: Cache-first strategy (check SW content if exists)
if sw_found:
    sw_content = read_file(sw_found)
    cache_first = bool(re.search(r'cache.*first|caches\.match|cache\.match', sw_content, re.IGNORECASE))
    stale_revalidate = bool(re.search(r'stale.*revalidate|networkFirst|cacheFirst', sw_content, re.IGNORECASE))
    record("F12-Q2", "Cache strategy in service worker",
           cache_first or stale_revalidate,
           f"cache-first={cache_first}, stale-while-revalidate={stale_revalidate}")
else:
    # Check for cache headers in server code instead
    cache_in_server = False
    for f in collect_files(SRC, "*.py", recursive=True):
        content = read_file(f)
        if re.search(r'Cache-Control|max-age|stale-while-revalidate', content, re.IGNORECASE):
            cache_in_server = True
            break
    record("F12-Q2", "Cache strategy exists (SW or server headers)",
           cache_in_server,
           f"No service worker; server cache headers: {cache_in_server}")

# Q3: Manifest.json for PWA
manifest_paths = [
    WEB / "manifest.json",
    WEB / "manifest.webmanifest",
    WEB / "site.webmanifest",
    BROWSER_ROOT / "manifest.json",
]
manifest_found = None
for p in manifest_paths:
    if p.exists():
        manifest_found = p
        break
# Check HTML for manifest link
manifest_link = []
for hf in collect_files(WEB, "*.html", recursive=True):
    content = read_file(hf)
    if 'manifest' in content.lower():
        links = re.findall(r'<link[^>]*rel=["\']manifest["\'][^>]*>', content, re.IGNORECASE)
        manifest_link.extend(links)
record("F12-Q3", "PWA manifest exists",
       manifest_found is not None or len(manifest_link) > 0,
       f"Manifest file: {manifest_found}, HTML links: {manifest_link[:3]}")

# Q4: Offline fallback
offline_page = WEB / "offline.html"
has_offline = offline_page.exists()
if sw_found:
    sw_content = read_file(sw_found)
    has_offline = has_offline or bool(re.search(r'offline', sw_content, re.IGNORECASE))
record("F12-Q4", "Offline fallback exists",
       has_offline,
       f"offline.html={offline_page.exists()}, SW offline handling={'yes' if sw_found else 'no SW'}")

# Q5: Cache versioning strategy
if sw_found:
    sw_content = read_file(sw_found)
    has_version = bool(re.search(r'CACHE_VERSION|cacheName|cache.*v[0-9]|version', sw_content, re.IGNORECASE))
    record("F12-Q5", "Cache versioning/invalidation strategy",
           has_version,
           f"Version pattern found in SW: {has_version}")
else:
    record("F12-Q5", "Cache versioning strategy", False,
           "No service worker found to check versioning")

In [ ]:
# ============================================================
# EVIDENCE SUMMARY -- Tower 17: Performance x Solace Browser
# ============================================================

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
finding_rate = (failed / total * 100) if total > 0 else 0

# Group by floor
floor_summary = {}
for r in results:
    floor = r["id"].split("-")[0]  # e.g. "F1"
    if floor not in floor_summary:
        floor_summary[floor] = {"passed": 0, "failed": 0, "total": 0}
    floor_summary[floor]["total"] += 1
    if r["status"] == "PASS":
        floor_summary[floor]["passed"] += 1
    else:
        floor_summary[floor]["failed"] += 1

floor_labels = {
    "F1": "Core Web Vitals", "F2": "JS Performance", "F3": "CSS Performance",
    "F4": "Network Optimizer", "F5": "Startup Time", "F6": "Memory Profiler",
    "F7": "Battery Life", "F8": "Rendering Pipeline", "F9": "Font Loading",
    "F10": "Image Optimization", "F11": "API Response", "F12": "Service Worker/Cache",
}

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "finding_rate": round(finding_rate, 1),
    "score": round(passed / total * 100, 1) if total > 0 else 0,
    "min_score": MIN_SCORE,
    "converged": finding_rate < 5,
    "floor_scores": {
        f"{k} ({floor_labels.get(k, '?')})": f"{v['passed']}/{v['total']}"
        for k, v in sorted(floor_summary.items())
    },
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME} -- SOLACE BROWSER QA")
print("=" * 70)
print(f"Total probes:  {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Score:         {summary['score']}%")
print(f"Finding rate:  {finding_rate:.1f}%")
print(f"Converged:     {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
print()
print("FLOOR BREAKDOWN:")
for k, v in sorted(floor_summary.items()):
    label = floor_labels.get(k, "?")
    pct = round(v["passed"] / v["total"] * 100) if v["total"] > 0 else 0
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    print(f"  {k:4s} {label:28s} {v['passed']}/{v['total']} [{bar}] {pct}%")

print()
if summary["findings"]:
    print(f"FINDINGS ({len(summary['findings'])}):\n")
    for f in summary["findings"]:
        print(f"  FAIL: {f['id']} {f['name']}")
        if f['detail']:
            print(f"        {f['detail'][:120]}")
else:
    print("ALL PROBES PASSED.")

print(f"\n--- Tower {TOWER} complete. Evidence sealed. ---")